# Proyecto Circuitos Hamiltonianos — Farmacias Simán SPS

Notebook organizado por etapas. A partir de esta version, los circuitos Hamiltonianos
se leen **directamente de la hoja `Candidatos Bloques`**, que ya contiene, para cada
combinacion Candidato x Bloque, el recorrido completo (arranca y termina en el candidato)
con sus distancias de ida y vuelta registradas en campo. Ya no se usa la hoja
`Distancias de Farmacias Vecinas` ni `Distancias de Candidatos`: el candidato ahora
SI forma parte del circuito, porque es el punto real de partida y llegada del tecnico.

**Estructura:**
1. Configuracion y datos
2. Datos de los candidatos (zona, ubicacion, area, precio de alquiler)
3. Carga de los recorridos por candidato y bloque (hoja `Candidatos Bloques`)
4. Construccion del subgrafo (circuito) por candidato y bloque
5. Verificacion de Dirac y Ore
6. Backtracking del circuito Hamiltoniano
7. Verificacion y distancia del circuito Hamiltoniano
8. Ejecutar verificacion y busqueda para los 3 candidatos x 5 bloques
9. Comparacion final: que candidato conviene mas y por que


## 1. Importaciones y funciones auxiliares

In [ ]:
from pathlib import Path
from math import inf
import unicodedata
import openpyxl

ARCHIVO_EXCEL = Path("Datos_SucursalesSimán.xlsx")

#convierte el valor del excel en float
def numero(valor):
    if valor is None:
        return None

    try:
        return float(valor)

    except (TypeError, ValueError):
        return None


def formato_km(valor):
    if valor is None or valor==inf:
        return "N/D"

    return f"{valor:.2f} km"


## 2. Datos de los candidatos

In [ ]:
CANDIDATOS={

    "A": {
        "nombre": "Candidato A",
        "zona": "Zona Norte",
        "ubicacion": "Sector La 105 Brigada",
        "area_m2": 450,
        "precio_usd": 3825,
    },
    "B": {
        "nombre": "Candidato B",
        "zona": "Zona Sur",
        "ubicacion": "Zona industrial, Sector el Polvorín",
        "area_m2": 764.22,
        "precio_usd": 5732,
    },
    "C": {
        "nombre": "Candidato C",
        "zona": "Zona Centro",
        "ubicacion": "Barrio Los Andes, 21102 San Pedro Sula, Cortés, Honduras",
        "area_m2": 75,
        "precio_usd": 1425,
    },

}


def nombre_candidato(letra):
    return CANDIDATOS[str(letra).upper()]["nombre"]

def datos_candidato(letra):
    return CANDIDATOS[str(letra).upper()]

if not ARCHIVO_EXCEL.exists():
    raise FileNotFoundError(f"No se encontro el archivo: {ARCHIVO_EXCEL.resolve()}")

wb=openpyxl.load_workbook(ARCHIVO_EXCEL, data_only=True)


## 3. Carga de los recorridos por candidato y bloque

Se lee la hoja **`Candidatos Bloques`**. Ahi, para cada Candidato (A, B, C) y cada
Bloque (1 al 5), ya esta registrado el circuito completo: la secuencia ordenada de
tramos ("Recorrido") con su Distancia Ida y Distancia Vuelta en km, empezando y
terminando en el candidato.

Se normalizan los nombres de sucursal (acentos/espacios) porque un mismo nodo
puede aparecer escrito de forma ligeramente distinta en distintas filas (por
ejemplo "Simán Zip San Jose" vs "Simán Zip San José"); sin esto, el algoritmo los
trataria como dos sucursales diferentes y el circuito no podria cerrar.

In [3]:
_alias_nombres={}

#normaliza el nombre de una sucursal para que variantes con/sin tilde se traten como el mismo nodo
def normalizar_nombre(nombre):
    nombre = str(nombre).strip()
    clave = "".join(c for c in unicodedata.normalize("NFKD", nombre) if not unicodedata.combining(c)).lower()

    if clave not in _alias_nombres:
        _alias_nombres[clave] = nombre

    return _alias_nombres[clave]


#Lee la hoja 'Candidatos Bloques' y arma, para cada candidato y cada bloque,
#la lista ordenada de tramos (origen, destino, ida, vuelta) tal como fue registrada en campo.
def cargar_recorridos_candidatos(hoja):

    recorridos = {"A": {1: [], 2: [], 3: [], 4: [], 5: []},
                  "B": {1: [], 2: [], 3: [], 4: [], 5: []},
                  "C": {1: [], 2: [], 3: [], 4: [], 5: []}}

    candidato_actual = None
    bloque_actual = None

    for fila in range(4, hoja.max_row + 1):
        cand_txt = hoja.cell(fila, 1).value
        bloque_txt = hoja.cell(fila, 2).value
        recorrido_txt = hoja.cell(fila, 3).value
        ida = numero(hoja.cell(fila, 4).value)
        vuelta = numero(hoja.cell(fila, 5).value)

        #las columnas Candidato y Bloque estan fusionadas: solo aparecen en la primera fila del grupo
        if cand_txt:
            candidato_actual = str(cand_txt).strip().split()[-1].upper()
        if bloque_txt:
            bloque_actual = int(str(bloque_txt).strip().split()[-1])

        if not recorrido_txt or candidato_actual is None or bloque_actual is None:
            continue

        if " → " not in str(recorrido_txt):
            continue

        origen, destino = str(recorrido_txt).split(" → ")
        origen, destino = normalizar_nombre(origen), normalizar_nombre(destino)

        #si en campo solo se registro un sentido, se asume la misma distancia en el otro
        if ida is None and vuelta is not None:
            ida = vuelta
        if vuelta is None and ida is not None:
            vuelta = ida

        recorridos[candidato_actual][bloque_actual].append((origen, destino, ida, vuelta))

    return recorridos


hoja_candidatos_bloques = wb["Candidatos Bloques"]
recorridos_candidatos = cargar_recorridos_candidatos(hoja_candidatos_bloques)

#validacion: cada bloque debe cerrar el circuito saliendo y regresando al mismo candidato
for letra, bloques_dict in recorridos_candidatos.items():
    for numero_bloque, aristas in bloques_dict.items():
        if not aristas:
            print(f"AVISO: Candidato {letra} - Bloque {numero_bloque} no tiene recorridos cargados.")
            continue

        if aristas[0][0] != f"Candidato {letra}" or aristas[-1][1] != f"Candidato {letra}":
            print(f"AVISO: Candidato {letra} - Bloque {numero_bloque} no cierra el circuito en el candidato.")


## 4. Construccion del subgrafo (circuito) por candidato y bloque

A diferencia de la version anterior (donde el candidato quedaba fuera del circuito y
el subgrafo se sacaba de un grafo general de sucursales vecinas), ahora el subgrafo de
cada Candidato x Bloque se construye **unicamente con las aristas de su propio
recorrido**, y el candidato **si es un vertice mas** del circuito, porque es el punto
real de partida y llegada del tecnico.

In [4]:
#Construye el subgrafo (no dirigido) del candidato y bloque indicados,
#usando las distancias ida/vuelta registradas en 'Candidatos Bloques'.
def construir_subgrafo(letra, numero_bloque):

    letra = str(letra).upper()
    aristas = recorridos_candidatos[letra][numero_bloque]
    subgrafo = {}

    def agregar_vertice(nombre):
        if nombre not in subgrafo:
            subgrafo[nombre] = {}

    for origen, destino, ida, vuelta in aristas:

        if ida is None or vuelta is None:
            continue

        promedio = (ida + vuelta) / 2
        agregar_vertice(origen)
        agregar_vertice(destino)

        datos = {"ida": ida, "vuelta": vuelta, "peso": promedio}

        subgrafo[origen][destino] = datos
        #A -> B
        #ida = 2.0
        #vuelta = 3.0

        #B -> A
        #ida = 3.0
        #vuelta = 2.0
        subgrafo[destino][origen] = {"ida": vuelta, "vuelta": ida, "peso": promedio}

    return subgrafo


## 5. Verificación de Dirac y Ore

# Definicion

1. **Dirac:**
Es una de las condiciones que nos ayuda a determinar si un grafo tiene circuito hamiltoniano. Eso si que un grafo no cumpla Dirac NO significa necesariamente que no sea hamiltoniano.

    El teorema dice: Para un grafo simple con n ≥ 3 vertices, si el grado de cada vertice cumple: deg(v)≥n/2

    En otras palabras: Si cada vertice esta conectado con al menos la mitad de los demas vertices, entonces existe un circuito hamiltoniano.

    Ejem: vertices(n)=6  6/2=3  -> para todos los vertices si tenemos A:4 | B: 3 | C:5 | D:3 | E:4 | F::3
    El grafo es un **circuito hamiltoniano**


2. **Ore:**
Ore utiliza una condicion diferente.

    Para un grafo simple con n≥3, si para cada par de vértices no adyacentes (u) y (v): deg(u)+deg(v) ≥ n
    Entonces el grafo tiene **circuito hamiltoniano**

    No adyacentes: Significa que no existe una arista entre ellos.

    Ejem: n=6. Para aplicar Ore necesitamos revisar los pares de vertices que no estan conectados.
    Suponiendo tenemos deg(A)=4 y deg(D)=3 y A y D  no estan conectados.
    Entonces  deg(A)+deg(D)=4+3=7 -> como 7≥6, se cumple la condicion la condicion para ese par.
    Hay que hacer esto con todos los pares de vértices no adyacentes.
    deg(u)+deg(v)≥n

    entonces:

    Existe un circuito hamiltoniano

    Si todos cumplen:
    deg(u)+deg(v)≥n
    Entonces **Existe un circuito hamiltoniano**

> **Nota sobre este caso:** como cada subgrafo ahora es exactamente el circuito ya
> trazado en campo (y no todas las conexiones posibles entre sucursales cercanas),
> cada vertice tiene grado 2. Con n=13 vertices por bloque, ni Dirac (2 >= 6.5) ni Ore
> se cumplen casi nunca — eso es esperado y no invalida el circuito: estos teoremas son
> condiciones **suficientes**, no necesarias. Por eso el paso decisivo sigue siendo el
> backtracking de la seccion 6, que confirma si el circuito reportado es valido.

In [5]:
#Calcula el grado de cada vertice del subgrafo.
def calcular_grados(subgrafo):
    grados = {}
    for vertice in subgrafo:
        grados[vertice] = len(subgrafo[vertice])

    return grados

#Verifica la condicion suficiente de Dirac. Un grafo simple con n >= 3 vertices cumple Dirac si:grado minimo >= n / 2
#si se cumple, el grafo es Hamiltoniano.
def verificar_dirac(subgrafo):
    n = len(subgrafo)
    grados = calcular_grados(subgrafo)

    if n < 3:
        return {"cumple": False, "n": n, "grado_minimo": min(grados.values()) if grados else 0}

    grado_minimo = min(grados.values())
    cumple = grado_minimo >= n / 2

    return {"cumple": cumple, "n": n, "grado_minimo": grado_minimo}

#Verifica la condicion suficiente de Ore.  Para todo par de vertices NO adyacentes:  grado(u) + grado(v) >= n
def verificar_ore(subgrafo):

    n = len(subgrafo)
    grados = calcular_grados(subgrafo)
    pares_evaluados = []
    cumple = True
    vertices = list(subgrafo.keys())
    for i in range(len(vertices)):
        for j in range(i + 1, len(vertices)):
            u = vertices[i]
            v = vertices[j]

            #solo interesa revisar pares no conectados
            if v not in subgrafo[u]:
                suma_grados = grados[u] + grados[v]

                pares_evaluados.append((u, v, grados[u], grados[v], suma_grados))

                if suma_grados < n:
                    cumple = False

    return {"cumple": cumple, "n": n, "pares": pares_evaluados}


def imprimir_linea():
    print("=" * 90)

def imprimir_verificacion_hamiltoniana(letra, numero_bloque):

    bloque = f"Bloque {numero_bloque}"
    subgrafo = construir_subgrafo(letra, numero_bloque)

    grados = calcular_grados(subgrafo)
    resultado_dirac = verificar_dirac(subgrafo)
    resultado_ore = verificar_ore(subgrafo)

    n_dirac = resultado_dirac["n"]
    grado_min = resultado_dirac["grado_minimo"]

    print("\n" + "=" * 90)
    print(f"VERIFICACIÓN HAMILTONIANA - {nombre_candidato(letra).upper()} - {bloque.upper()}")
    print("=" * 90)

    print("\nSucursales del circuito (incluye al candidato):")

    for vertice in sorted(grados):
        grado_v = grados[vertice]
        print(f"  {vertice}: grado = {grado_v}")

    print("\n" + "-" * 90)
    print("TEOREMA DE DIRAC")
    print("-" * 90)

    print(f"Numero de vertices: {n_dirac}")
    print(f"Grado minimo:       {grado_min}")
    print(f"n / 2:              {n_dirac / 2:.1f}")

    if resultado_dirac["cumple"]:
        print("Resultado: CUMPLE LA CONDICION DE DIRAC")
    else:
        print("Resultado: NO CUMPLE LA CONDICION DE DIRAC")
        print("Esto NO significa que no exista un circuito Hamiltoniano.")

    print("\n" + "-" * 90)
    print("TEOREMA DE ORE")
    print("-" * 90)

    if resultado_ore["cumple"]:
        print("Resultado: CUMPLE LA CONDICION DE ORE")
    else:
        print("Resultado: NO CUMPLE LA CONDICION DE ORE")
        print("Esto NO significa que no exista un circuito Hamiltoniano.")


## 6. Backtracking para el circuito Hamiltoniano

In [6]:
# Busca un circuito Hamiltoniano utilizando backtracking.

#    Un circuito Hamiltoniano debe:
#    1. Visitar todos los vértices exactamente una vez.
#    2. No repetir vertices durante el recorrido.
#    3. Regresar al vértice inicial.

def buscar_circuito_hamiltoniano(subgrafo):
    vertices = list(subgrafo.keys())
    if len(vertices) < 3:
        return []

    #se elige una farmacia como punto inicial
    inicio = vertices[0]
    ruta = [inicio]
    visitados = {inicio}


    def backtracking(actual):

        # Si ya visitamos todos los vértices,comprobamos si podemos regresar al inicio.
        if len(ruta) == len(vertices):
            if inicio in subgrafo[actual]:
                return True

            return False

        #se revisa los vecinos del vertice actual
        for vecino in subgrafo[actual]:

            #si la farmacia todavia no ha sido visitada
            if vecino not in visitados:

                #agregar a la ruta
                ruta.append(vecino)
                visitados.add(vecino)

                #continuar buscando
                if backtracking(vecino):
                    return True

                #si esta opcion no funciona, se deshace la decision
                ruta.pop()
                visitados.remove(vecino)

        return False

    #iniciar busqueda
    encontrado = backtracking(inicio)

    if encontrado:
        #se cierra el circuito regresando al inicio
        ruta.append(inicio)
        return ruta

    return []


## 7. Verificación y distancia del circuito Hamiltoniano

In [7]:
#Verifica que un circuito encontrado sea Hamiltoniano.
def verificar_circuito_hamiltoniano(circuito, subgrafo):

    if not circuito:
        return False

    #El primer y ultimo vértice deben ser iguales
    if circuito[0] != circuito[-1]:
        return False

    #Quito el ultimo elemento para contar solamente los vertices visitados.
    vertices_visitados = circuito[:-1]

    #Todos los vertices deben ser diferentes
    if len(vertices_visitados) != len(set(vertices_visitados)):
        return False

    #se debe visitar todos los vertices del bloque (incluye al candidato)
    if set(vertices_visitados) != set(subgrafo.keys()):
        return False

    #se verifica que cada conexion exista
    for actual, siguiente in zip(circuito, circuito[1:]):
        if siguiente not in subgrafo[actual]:
            return False

    return True


#  Calcula la distancia total de ida, vuelta y promedio del circuito Hamiltoniano.
def calcular_distancia_circuito(circuito, subgrafo):

    total_ida = 0.0
    total_vuelta = 0.0

    for actual, siguiente in zip(circuito, circuito[1:]):
        datos = subgrafo[actual][siguiente]
        total_ida += datos["ida"]
        total_vuelta += datos["vuelta"]

    promedio = (total_ida + total_vuelta) / 2

    return total_ida, total_vuelta, promedio


def analizar_circuito(letra, numero_bloque):

    bloque = f"Bloque {numero_bloque}"
    subgrafo = construir_subgrafo(letra, numero_bloque)
    circuito = buscar_circuito_hamiltoniano(subgrafo)

    print("\n" + "=" * 90)
    print(f"{nombre_candidato(letra).upper()} - {bloque.upper()}")
    print("=" * 90)

    if not circuito:
        print("\nNo se encontro un circuito Hamiltoniano.")
        return {"circuito": [], "valido": False}

    valido = verificar_circuito_hamiltoniano(circuito, subgrafo)

    print("\nCircuito encontrado:")
    print(" -> ".join(circuito))

    print("\nCantidad de sucursales visitadas (sin contar el candidato):")
    print(len(circuito) - 2)

    print("\n¿Es un circuito Hamiltoniano valido?")
    print("SÍ" if valido else "NO")

    resultado = {"circuito": circuito, "valido": valido, "subgrafo": subgrafo}

    if valido:
        ida, vuelta, promedio = calcular_distancia_circuito(circuito, subgrafo)

        print("\nDistancia del circuito:")
        print(f"  Ida:      {ida:.2f} km")
        print(f"  Vuelta:   {vuelta:.2f} km")
        print(f"  Promedio: {promedio:.2f} km")

        resultado.update({"ida": ida, "vuelta": vuelta, "promedio": promedio})

    return resultado


## 8. Ejecutar verificación y búsqueda para los 3 candidatos × 5 bloques

Primero verificamos Dirac y Ore, y despues buscamos y validamos el circuito Hamiltoniano
mediante backtracking, para cada uno de los 15 subgrafos (3 candidatos x 5 bloques).

In [8]:
resultados = {"A": {}, "B": {}, "C": {}}

for letra in ["A", "B", "C"]:
    for numero_bloque in range(1, 6):
        imprimir_verificacion_hamiltoniana(letra, numero_bloque)
        resultados[letra][numero_bloque] = analizar_circuito(letra, numero_bloque)



VERIFICACIÓN HAMILTONIANA - CANDIDATO A - BLOQUE 1

Sucursales del circuito (incluye al candidato):
  33 Calle Col Guillen: grado = 2
  Candidato A: grado = 2
  SIMÁN PLAZA VICAY: grado = 2
  Simán Arenales: grado = 2
  Simán Calpules: grado = 2
  Simán Calpules 2: grado = 2
  Simán Ivihsa: grado = 2
  Simán Luisiana: grado = 2
  Simán Pradera: grado = 2
  Simán Santa Martha: grado = 2
  Simán Zip San José: grado = 2
  Town Center: grado = 2
  VILLA OLIMPICA: grado = 2

------------------------------------------------------------------------------------------
TEOREMA DE DIRAC
------------------------------------------------------------------------------------------
Numero de vertices: 13
Grado minimo:       2
n / 2:              6.5
Resultado: NO CUMPLE LA CONDICION DE DIRAC
Esto NO significa que no exista un circuito Hamiltoniano.

------------------------------------------------------------------------------------------
TEOREMA DE ORE
------------------------------------------------

## Resumen de Dirac, Ore y validez del circuito

Tabla compacta con los 15 subgrafos (3 candidatos x 5 bloques): si cumplen Dirac,
si cumplen Ore, si el backtracking encontro un circuito Hamiltoniano valido y su
distancia promedio.

In [9]:
encabezado = f"{'Candidato':<11}{'Bloque':<9}{'Dirac':<8}{'Ore':<8}{'Circuito valido':<18}{'Distancia prom.':<16}"
print(encabezado)
print("-" * 70)

sin_dirac_ni_ore = 0
total_subgrafos = 0

for letra in ["A", "B", "C"]:
    for numero_bloque in range(1, 6):
        bloque = f"Bloque {numero_bloque}"
        subgrafo = construir_subgrafo(letra, numero_bloque)
        dirac = verificar_dirac(subgrafo)
        ore = verificar_ore(subgrafo)
        resultado = resultados[letra][numero_bloque]

        total_subgrafos += 1
        if not dirac["cumple"] and not ore["cumple"]:
            sin_dirac_ni_ore += 1

        valido_txt = "Sí" if resultado["valido"] else "No"
        dist = formato_km(resultado["promedio"]) if resultado["valido"] else "N/D"
        dirac_txt = "Sí" if dirac["cumple"] else "No"
        ore_txt = "Sí" if ore["cumple"] else "No"

        fila = f"{letra:<11}{bloque:<9}{dirac_txt:<8}{ore_txt:<8}{valido_txt:<18}{dist:<16}"
        print(fila)

print()
print(f"Interpretación: {sin_dirac_ni_ore} de {total_subgrafos} subgrafos no cumplen ni Dirac ni Ore, "
      f"porque cada circuito registrado en campo es una ruta cerrada simple (grado 2 en cada nodo). "
      f"Dirac y Ore son condiciones SUFICIENTES, no necesarias: el backtracking confirmo igualmente "
      f"la validez de los circuitos donde fue posible cerrarlos con los datos de campo.")


Candidato  Bloque   Dirac   Ore     Circuito valido   Distancia prom. 
----------------------------------------------------------------------
A          Bloque 1 No      No      Sí                35.12 km        
A          Bloque 2 No      No      Sí                18.48 km        
A          Bloque 3 No      No      Sí                25.12 km        
A          Bloque 4 No      No      Sí                30.95 km        
A          Bloque 5 No      No      Sí                16.05 km        
B          Bloque 1 No      No      Sí                27.30 km        
B          Bloque 2 No      No      Sí                31.21 km        
B          Bloque 3 No      No      Sí                26.27 km        
B          Bloque 4 No      No      Sí                45.90 km        
B          Bloque 5 No      No      Sí                21.35 km        
C          Bloque 1 No      No      Sí                34.17 km        
C          Bloque 2 No      No      Sí                18.99 km        
C     

## 9. Comparación final: ¿qué candidato conviene más y por qué?

Para cada candidato se suma la distancia promedio (ida+vuelta ÷ 2) de sus 5 circuitos
Hamiltonianos: esa suma representa cuantos km reales tendria que recorrer un tecnico
saliendo de esa sede para revisar, en cada bloque, todas las sucursales una sola vez y
volver al punto de partida. Se compara esa cifra junto con el costo y el area de
alquiler de cada candidato, para dar una recomendacion que combine el criterio
matematico (distancia) con el criterio logistico-operativo (costo/espacio).

In [11]:
def comparar_candidato(letra):
    """Muestra, ordenados de menor a mayor distancia, los 5 circuitos Hamiltonianos
    del candidato, y devuelve la suma total de sus distancias promedio."""

    info = datos_candidato(letra)
    nombre = info["nombre"]
    zona = info["zona"]
    ubicacion = info["ubicacion"]
    area = info["area_m2"]
    precio = info["precio_usd"]

    filas = []
    for numero_bloque in range(1, 6):
        r = resultados[letra][numero_bloque]
        if r["valido"]:
            filas.append((numero_bloque, r))

    filas.sort(key=lambda item: item[1]["promedio"])

    imprimir_linea()
    print(f"{nombre.upper()} — {zona.upper()}")
    print(f"Ubicación: {ubicacion}")
    print(f"Área: {area} m²   |   Alquiler: ${precio:,}")
    imprimir_linea()

    suma_total = 0.0
    for numero_bloque, r in filas:
        n_sucursales = len(r["circuito"]) - 2
        print(f"\nBloque {numero_bloque}  ({n_sucursales} sucursales)")
        print(f"   Ida:      {formato_km(r['ida'])}")
        print(f"   Vuelta:   {formato_km(r['vuelta'])}")
        print(f"   Promedio: {formato_km(r['promedio'])}")
        suma_total += r["promedio"]

    if filas:
        num_cerca, r_cerca = filas[0]
        num_lejos, r_lejos = filas[-1]
        print(f"\nBloque más económico para {nombre}: Bloque {num_cerca} ({formato_km(r_cerca['promedio'])})")
        print(f"Bloque más costoso para {nombre}:  Bloque {num_lejos} ({formato_km(r_lejos['promedio'])})")

    print(f"\nSuma total recorrida por {nombre} en sus 5 circuitos: {formato_km(suma_total)}")

    return suma_total


## 10. Ejecutar la comparación y elegir el candidato más conveniente

In [12]:
sumas = {}

for letra in ["A", "B", "C"]:
    sumas[letra] = comparar_candidato(letra)
    print()

imprimir_linea()
print("COMPARACIÓN FINAL DE LOS 3 CANDIDATOS")
imprimir_linea()

encabezado = f"\n{'Candidato':<12}{'Km totales':<14}{'Alquiler':<12}{'Área (m²)':<12}"
print(encabezado)
print("-" * 50)
for letra, suma in sorted(sumas.items(), key=lambda item: item[1]):
    info = datos_candidato(letra)
    precio_txt = f"${info['precio_usd']:,}"
    fila = f"{letra:<12}{formato_km(suma):<14}{precio_txt:<12}{info['area_m2']:<12}"
    print(fila)

mejor = min(sumas, key=sumas.get)
peor = max(sumas, key=sumas.get)
ahorro_km = sumas[peor] - sumas[mejor]

print()
imprimir_linea()
print("RECOMENDACIÓN FINAL")
imprimir_linea()

nombre_mejor = nombre_candidato(mejor)
nombre_peor = nombre_candidato(peor)

print(f"\nDe los 15 subgrafos evaluados (3 candidatos x 5 bloques), a NetGuard Solutions le conviene")
print(f"más operar desde el {nombre_mejor}: es el que recorre MENOS kilómetros en total")
print(f"({formato_km(sumas[mejor])}) al visitar las sucursales de los 5 bloques, {ahorro_km:.2f} km menos")
print(f"que la peor opción ({nombre_peor}, {formato_km(sumas[peor])}).")

info_mejor = datos_candidato(mejor)
print(f"\nAdemás de la ventaja de distancia, {nombre_mejor} ({info_mejor['zona']}) cuesta")
print(f"${info_mejor['precio_usd']:,} de alquiler con {info_mejor['area_m2']} m², lo que se debe sopesar junto")
print(f"con el argumento logístico: menos km por bloque significa menos tiempo de despacho,")
print(f"menos desgaste de vehículo y una respuesta más rápida ante fallas de los sensores IoT.")


CANDIDATO A — ZONA NORTE
Ubicación: Sector La 105 Brigada
Área: 450 m²   |   Alquiler: $3,825

Bloque 5  (12 sucursales)
   Ida:      14.30 km
   Vuelta:   17.80 km
   Promedio: 16.05 km

Bloque 2  (12 sucursales)
   Ida:      19.05 km
   Vuelta:   17.92 km
   Promedio: 18.48 km

Bloque 3  (12 sucursales)
   Ida:      23.35 km
   Vuelta:   26.90 km
   Promedio: 25.12 km

Bloque 4  (12 sucursales)
   Ida:      26.20 km
   Vuelta:   35.70 km
   Promedio: 30.95 km

Bloque 1  (12 sucursales)
   Ida:      29.75 km
   Vuelta:   40.50 km
   Promedio: 35.12 km

Bloque más económico para Candidato A: Bloque 5 (16.05 km)
Bloque más costoso para Candidato A:  Bloque 1 (35.12 km)

Suma total recorrida por Candidato A en sus 5 circuitos: 125.73 km

CANDIDATO B — ZONA SUR
Ubicación: Zona industrial, Sector el Polvorín
Área: 764.22 m²   |   Alquiler: $5,732

Bloque 5  (12 sucursales)
   Ida:      19.70 km
   Vuelta:   23.00 km
   Promedio: 21.35 km

Bloque 3  (12 sucursales)
   Ida:      25.35 km
   